# Ablation ở n=500 - đưa đường cong $k$ ra khỏi vùng nhiễu

Ablation trong báo cáo đang ở **n=50**, nơi ngưỡng phát hiện ghép cặp là
**18.0 F1**. Khoảng cách của đường cong $k$ là **5.7 F1**, tức nằm gọn trong
nhiễu - báo cáo đã ghi thẳng là không kết luận được.

Ở **n=500** ngưỡng xuống **~4.9 F1**, nhỏ hơn 5.7, nên phép đo này *có thể*
phân giải được. Đó là lý do chọn n=500 chứ không phải n=200 (ngưỡng 7.7, vẫn
lớn hơn 5.7 → chạy cũng vô ích).

**Chỉ chạy 10/26 cấu hình.** Đo thật cho 2.90 s/câu, nên cả 26 cấu hình ở
n=500 tốn ~10.5h - vượt một phiên. Hai trục được chọn:

| trục | vì sao |
|---|---|
| `percentile` (5) | đường cong $k$ - khẳng định đang nằm trong nhiễu |
| `max_iter` (5) | báo cáo vừa ghi đây là kết luận **phụ thuộc LLM vòng lặp**; n lớn hơn kiểm được nó có bền không |

`scorer` (4 cấu hình) **cố ý bỏ**: khoảng cách lexical–dual là 2.4 F1, vẫn dưới
ngưỡng 4.9 ở n=500, nên chạy cũng không kết luận được. Cần n≈1500.

In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn bằng cách clone repo. Trước đây repo còn private nên notebook
# phải tải qua một gist công khai; repo public rồi, nên clone thẳng là nguồn duy
# nhất, không còn hai bản phải giữ đồng bộ.
import os, sys, subprocess, shutil

BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'ANLP-IterCOMP')
GIT_URL = 'https://github.com/nhantrnh/ANLP-IterCOMP.git'
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data', 'metrics', 'fertility', 'stats', 'budget']

os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]

subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO], check=True)

miss = [m for m in NEED
        if not os.path.exists(os.path.join(REPO, 'src', 'itercomp', m + '.py'))]
if miss:
    raise SystemExit('Bản clone thiếu module: ' + ', '.join(miss))

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

sha = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                     capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| src/scripts/:', os.path.isdir('src/scripts'))
print('đủ', len(NEED), 'module | commit:', sha)


In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ ABLATION n=500, 10 cấu hình ══ ~4h trên T4
import subprocess, sys, os, time
READER='Qwen/Qwen2.5-7B-Instruct'; OUT='results/ablation_vimqa500_7b.json'
os.makedirs('results', exist_ok=True)

def run_stream(cmd, logfile):
    t0=time.time()
    with open(logfile,'w') as lf:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                           text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

if os.path.exists(OUT):
    print(f'[bỏ qua] đã có {OUT}')
else:
    rc=run_stream([sys.executable,'-u','scripts/run_ablation.py',
        '--dataset','vimqa','--limit','500','--llm','hf',
        '--reader','hf','--reader-model',READER,'--load-4bit',
        '--scorer','dual','--only','percentile,max_iter','--out',OUT],
        'results/log_ablation500.txt')
    assert rc==0, f'thất bại (rc={rc})'

In [ ]:
import shutil, os
BASE='/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z=shutil.make_archive(os.path.join(BASE,'results_ablation500'),'zip','results')
print('✓',z, f'({os.path.getsize(z)/1e6:.1f} MB)')